In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from tqdm.notebook import tqdm

# =============================================================================
# CONFIGURATION
# =============================================================================
# Paths
DATA_DIR = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")
FEATURES_FILE = DATA_DIR / "features_mlcrowd" / "features_master.pkl"
CRSP_FOLDER = Path(r"D:\OnlineData\Dropbox\CRSP")
OUTPUT_FOLDER = Path(r"C:\Users\willi\.vscode\Github\Data")

# Create output folder
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Horizons for abnormal returns
HORIZONS = [1, 3, 5, 10, 21, 42, 63]

print(f"Features File: {FEATURES_FILE}")
print(f"CRSP Folder: {CRSP_FOLDER}")
print(f"Output Folder: {OUTPUT_FOLDER}")

In [ ]:
# =============================================================================
# LOAD FEATURES
# =============================================================================
print(f"\nLoading master feature file...")
if not FEATURES_FILE.exists():
    raise FileNotFoundError(f"Features file not found: {FEATURES_FILE}")

df_features_master = pd.read_pickle(FEATURES_FILE)
df_features_master['date'] = pd.to_datetime(df_features_master['date'])
print(f"Loaded {len(df_features_master):,} rows.")
print(f"Date range: {df_features_master['date'].min()} to {df_features_master['date'].max()}")

# =============================================================================
# PROCESS YEARS
# =============================================================================
print(f"\n{'='*60}")
print("Starting year-by-year merge with abnormal return calculations...")
print(f"{'='*60}\n")

merge_stats = []
years = sorted(df_features_master['date'].dt.year.unique())

for year in tqdm(years, desc="Processing years"):
    # 1. Filter features for the current year
    df_st = df_features_master[df_features_master['date'].dt.year == year].copy()
    
    if df_st.empty:
        print(f"  Skipping {year}: No feature data")
        continue
        
    # 2. Load CRSP data
    crsp_file = CRSP_FOLDER / f"dsf_final_{year}.pkl"
    if not crsp_file.exists():
        print(f"  Skipping {year}: CRSP file not found ({crsp_file})")
        continue
        
    df_crsp = pd.read_pickle(crsp_file)
    df_crsp['date'] = pd.to_datetime(df_crsp['date'])
    
    # --- Calculate Abnormal Returns (from user snippet) ---
    
    # 1. DGTW: Abnormal return = cumulative return - benchmark return
    for h in HORIZONS:
        cumret_col = f'f_cumret{h}'
        dgtw_col = f'f_dgtw_ret{h}'
        if cumret_col in df_crsp.columns and dgtw_col in df_crsp.columns:
            df_crsp[f'ar_dgtw_{h}'] = df_crsp[cumret_col] - df_crsp[dgtw_col]
    
    # 2. CAPM, FF3, FF5, FF6: Cumulative abnormal returns
    models = ['capm', 'FF3', 'FF5', 'FF6']
    
    for model in models:
        # First, ensure we have all individual abnormal returns up to max horizon
        max_horizon = max(HORIZONS)
        ar_cols = []
        for i in range(1, max_horizon + 1):
            col = f'f_{model}_ar{i}'
            if col in df_crsp.columns:
                ar_cols.append(col)
        
        # Calculate cumulative abnormal returns for each horizon
        if ar_cols:
            for h in HORIZONS:
                # Sum from day 1 to day h
                cols_to_sum = [f'f_{model}_ar{i}' for i in range(1, h + 1) 
                              if f'f_{model}_ar{i}' in df_crsp.columns]
                if cols_to_sum:
                    df_crsp[f'ar_{model}_{h}'] = df_crsp[cols_to_sum].sum(axis=1)
    
    # Select columns for merge
    cols_to_keep = ['permno', 'ticker', 'date', 'f_cumret1']
    
    # Add all abnormal return columns
    for model in ['dgtw', 'capm', 'FF3', 'FF5', 'FF6']:
        for h in HORIZONS:
            ar_col = f'ar_{model}_{h}'
            if ar_col in df_crsp.columns:
                cols_to_keep.append(ar_col)
    
    df_crsp_subset = df_crsp[cols_to_keep].copy()
    
    # Convert ticker to uppercase for matching
    df_crsp_subset['ticker'] = df_crsp_subset['ticker'].str.upper()
    
    # Merge on ticker (symbol) and date
    # Note: features_master has 'symbol', CRSP has 'ticker'
    df_merged = pd.merge(
        df_crsp_subset,
        df_st,
        left_on=['ticker', 'date'],
        right_on=['symbol', 'date'],
        how='left'
    )
    
    # Drop the redundant symbol column
    if 'symbol' in df_merged.columns:
        df_merged = df_merged.drop(columns=['symbol'])
    
    # Save merged data
    output_file = OUTPUT_FOLDER / f"merged_{year}.pkl"
    df_merged.to_pickle(output_file)
    
    # Track statistics
    match_rate = (len(df_merged) / len(df_st)) * 100
    ar_cols_in_merge = [col for col in df_merged.columns if col.startswith('ar_')]
    
    merge_stats.append({
        'year': year,
        'features_rows': len(df_st),
        'crsp_rows': len(df_crsp),
        'merged_rows': len(df_merged),
        'matched_rows': df_merged['permno'].notna().sum(),
        'match_rate': match_rate,
        'ar_columns': len(ar_cols_in_merge)
    })
    
    print(f"  {year}: {len(df_st):,} rows -> {df_merged['permno'].notna().sum():,} matched ({match_rate:.1f}%), {len(ar_cols_in_merge)} AR columns")

# Summary statistics
print(f"\n{'='*60}")
print("Merge Summary")
print(f"{'='*60}\n")

if merge_stats:
    df_stats = pd.DataFrame(merge_stats)
    print(df_stats.to_string(index=False))

    print(f"\nOverall statistics:")
    print(f"  Total Feature rows: {df_stats['features_rows'].sum():,}")
    print(f"  Total matched rows: {df_stats['matched_rows'].sum():,}")
    if df_stats['features_rows'].sum() > 0:
        print(f"  Overall match rate: {(df_stats['matched_rows'].sum() / df_stats['features_rows'].sum() * 100):.2f}%")
else:
    print("No data processed.")

print(f"\nMerged files saved to: {OUTPUT_FOLDER}")

In [ ]:
# =============================================================================
# CONSOLIDATE AND CLEANUP
# =============================================================================
print(f"\n{'='*60}")
print("Consolidating yearly files...")
print(f"{'='*60}\n")

merged_files = sorted(list(OUTPUT_FOLDER.glob("merged_*.pkl")))
merged_files = merged_files[:-1]

if not merged_files:
    print("No merged files found to consolidate.")
else:
    # Load and concatenate all yearly files
    dfs = []
    for file_path in tqdm(merged_files, desc="Loading files"):
        dfs.append(pd.read_pickle(file_path))
    
    df_final = pd.concat(dfs, ignore_index=True)
    del dfs # Free memory

In [ ]:
# Define fill values for different column types
fill_values = {}

# 1. Volume and Counts (Fill with 0)
vol_cols = [c for c in df_final.columns if any(x in c for x in ['volume', 'count', 'n_bullish', 'n_bearish', 'total_labeled'])]
for c in vol_cols:
    fill_values[c] = 0

# 2. Sentiment Features (Fill with 0 - Neutral/No Signal)
sent_cols = [c for c in df_final.columns if 'sentiment' in c]
for c in sent_cols:
    fill_values[c] = 0

# 3. Abnormal/Deviation Features (Fill with 0)
abn_cols = [c for c in df_final.columns if 'abn' in c or 'abnormal' in c]
for c in abn_cols:
    fill_values[c] = 0

# 4. Binary Flags (Fill with 0)
flag_cols = [c for c in df_final.columns if 'extreme' in c or 'surge' in c]
for c in flag_cols:
    fill_values[c] = 0

# 5. Special Cases
if 'disagreement_index' in df_final.columns:
    fill_values['disagreement_index'] = 0  # No disagreement if no data
if 'attention_hhi' in df_final.columns:
    fill_values['attention_hhi'] = 0       # No concentration
if 'silence_gap_hours' in df_final.columns:
    # Fill with a large value (e.g., 24 hours) to indicate silence
    fill_values['silence_gap_hours'] = 120
if 'intraday_sentiment_volatility' in df_final.columns:
    fill_values['intraday_sentiment_volatility'] = 0  # No volatility if no data

# 6. Features_05 (Sentiment Dynamics & Cohorts) -- Fill with 0 = no signal / not observed.
# These columns aren't caught by the generic patterns above (no tagged posts that
# symbol-day means no flips, no conviction streak, no cohort activity -- 0 is the
# correct value, not just an imputed placeholder).
f05_patterns = ['to_bull', 'to_bear', 'flip', 'conviction_index',
                'first_mover_net_sent', 'fresh_blood_ratio', 'whale_dominance',
                'minnow_dominance', 'night_owl_ratio', 'day_trader_ratio',
                'retention_rate', 'crowding_gini', 'specialist_ratio']
f05_cols = [c for c in df_final.columns if any(p in c for p in f05_patterns)]
for c in f05_cols:
    fill_values[c] = 0

# 7. Features_08 (Text-Embedding Signal) -- Fill with 0 = no signal / not observed.
# text_signal_mean/std are only observed once the walk-forward model has seen prior
# years of data (the cold-start year is NaN by design) and only for stock-days with
# StockTwits messages that day; text_signal_n is a message count. 0 is consistent with
# the "no signal" convention used for the other crowd features above.
f08_cols = [c for c in df_final.columns if c.startswith('text_signal_')]
for c in f08_cols:
    fill_values[c] = 0

# Apply imputation
print(f"Imputing missing values...")
df_final = df_final.fillna(value=fill_values)

# Check if any NaNs remain
remaining_nans = df_final.isnull().sum()[df_final.isnull().sum() > 0]
if not remaining_nans.empty:
    print(f"\nWarning: Some columns still have NaNs:")
    print(remaining_nans)
else:
    print(f"✓ All missing values filled.")

In [ ]:
# Save master file
master_output_file = OUTPUT_FOLDER / "merged_master.pkl"
print(f"Saving master file to: {master_output_file}")
df_final.to_pickle(master_output_file)
print(f"Saved {len(df_final):,} rows.")

# Delete intermediate files
print("Deleting intermediate yearly files...")
for file_path in merged_files:
    try:
        file_path.unlink()
    except Exception as e:
        print(f"Error deleting {file_path.name}: {e}")
        
print("Cleanup complete.")